# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 06: Observability with Error Tables & Monitoring

---

### What You'll Learn
- Enabling **DML Error Logging** on tables
- Loading intentionally bad data to generate errors
- Querying **Error Tables** for diagnostics
- Setting up **Alerts** for error thresholds
- Building monitoring patterns for production pipelines

### Error Tables Overview
Error Tables capture row-level DML errors (INSERT, COPY, MERGE) without failing the entire operation. This enables:
- **Continue-on-error** ingestion (load good rows, log bad ones)
- **Root cause analysis** (which rows failed and why)
- **Alerting** (notify when error rates exceed thresholds)

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Create a Table with Constraints (for error generation)

In [ ]:
-- =============================================================================
-- CREATE TABLE WITH CONSTRAINTS (errors will occur when violated)
-- =============================================================================

CREATE OR REPLACE TABLE TRANSACTIONS_VALIDATED (
    TXN_ID VARCHAR NOT NULL,
    ACCOUNT_ID VARCHAR NOT NULL,
    TXN_DATE TIMESTAMP NOT NULL,
    TXN_TYPE VARCHAR NOT NULL,
    AMOUNT NUMBER(12,2) NOT NULL,
    CURRENCY VARCHAR(3) DEFAULT 'USD',
    MERCHANT VARCHAR,
    CATEGORY VARCHAR,
    CHANNEL VARCHAR,
    STATUS VARCHAR,
    REFERENCE_NUM VARCHAR,
    DESCRIPTION VARCHAR,
    _LOADED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
-- Enable error logging on this table
ENABLE_ERROR_TABLE = TRUE
COMMENT = 'Transaction table with constraints and error logging enabled';

## Step 2: Load Good Data Successfully

In [ ]:
-- Load good data from existing transactions (should succeed with 0 errors)
INSERT INTO TRANSACTIONS_VALIDATED (
    TXN_ID, ACCOUNT_ID, TXN_DATE, TXN_TYPE, AMOUNT, CURRENCY,
    MERCHANT, CATEGORY, CHANNEL, STATUS, REFERENCE_NUM, DESCRIPTION
)
SELECT 
    TXN_ID, ACCOUNT_ID, TO_TIMESTAMP(TXN_DATE), TXN_TYPE, AMOUNT, CURRENCY,
    MERCHANT, CATEGORY, CHANNEL, STATUS, REFERENCE_NUM, DESCRIPTION
FROM TRANSACTIONS
LIMIT 10000;

SELECT COUNT(*) AS loaded_rows FROM TRANSACTIONS_VALIDATED;

## Step 3: Intentionally Load Bad Data (to trigger error logging)

We'll insert rows that violate NOT NULL constraints and have type mismatches.

In [ ]:
-- =============================================================================
-- INSERT BAD DATA (will partially fail - errors go to error table)
-- =============================================================================

-- Attempt 1: NULL values in NOT NULL columns
INSERT INTO TRANSACTIONS_VALIDATED (
    TXN_ID, ACCOUNT_ID, TXN_DATE, TXN_TYPE, AMOUNT, MERCHANT
)
VALUES 
    ('BAD_001', 'ACCT00000001', CURRENT_TIMESTAMP(), 'DEBIT', 100.00, 'Good Row'),
    (NULL, 'ACCT00000002', CURRENT_TIMESTAMP(), 'CREDIT', 50.00, 'Null TXN_ID'),        -- NULL PK
    ('BAD_003', NULL, CURRENT_TIMESTAMP(), 'DEBIT', 75.00, 'Null Account'),              -- NULL ACCOUNT_ID
    ('BAD_004', 'ACCT00000004', NULL, 'DEBIT', 200.00, 'Null Date'),                     -- NULL TXN_DATE
    ('BAD_005', 'ACCT00000005', CURRENT_TIMESTAMP(), NULL, 300.00, 'Null Type'),         -- NULL TXN_TYPE
    ('BAD_006', 'ACCT00000006', CURRENT_TIMESTAMP(), 'DEBIT', NULL, 'Null Amount'),      -- NULL AMOUNT
    ('BAD_007', 'ACCT00000007', CURRENT_TIMESTAMP(), 'CREDIT', 150.00, 'Good Row 2');

-- Note: With error tables enabled, good rows are inserted and bad rows are logged

In [ ]:
-- Attempt 2: Type conversion errors (string in numeric field)
INSERT INTO TRANSACTIONS_VALIDATED (
    TXN_ID, ACCOUNT_ID, TXN_DATE, TXN_TYPE, AMOUNT, MERCHANT
)
SELECT 
    'TYPE_ERR_' || SEQ4(),
    'ACCT00000010',
    CURRENT_TIMESTAMP(),
    'DEBIT',
    CASE WHEN MOD(SEQ4(), 3) = 0 THEN NULL ELSE SEQ4() * 10.50 END,  -- Some NULLs
    'Type Error Test'
FROM TABLE(GENERATOR(ROWCOUNT => 20));

## Step 4: Query the Error Table

The error table automatically captures all failed rows with diagnostic information.

In [ ]:
-- =============================================================================
-- QUERY ERROR TABLE - See what failed and why
-- =============================================================================

-- The error table is automatically created with naming convention:
-- <schema>.<table_name>_ERRORS

SELECT *
FROM SNOWFLAKE.CORE.GET_ERRORS('JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TRANSACTIONS_VALIDATED')
ORDER BY ERROR_TIMESTAMP DESC
LIMIT 20;

In [ ]:
-- Analyze error patterns
SELECT 
    ERROR_CODE,
    ERROR_MESSAGE,
    COLUMN_NAME,
    COUNT(*) AS ERROR_COUNT
FROM SNOWFLAKE.CORE.GET_ERRORS('JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TRANSACTIONS_VALIDATED')
GROUP BY ERROR_CODE, ERROR_MESSAGE, COLUMN_NAME
ORDER BY ERROR_COUNT DESC;

## Step 5: Set Up Alerting on Error Thresholds

Create an alert that fires when too many errors occur within a time window.

In [ ]:
-- =============================================================================
-- ALERT: Fire when error count exceeds threshold
-- =============================================================================

CREATE OR REPLACE ALERT HOL_ERROR_THRESHOLD_ALERT
    WAREHOUSE = HOL_XS_WH
    SCHEDULE = '5 MINUTE'
    COMMENT = 'Alert when DML errors exceed 10 in the last 5 minutes'
    IF (EXISTS (
        SELECT 1
        FROM SNOWFLAKE.CORE.GET_ERRORS('JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TRANSACTIONS_VALIDATED')
        WHERE ERROR_TIMESTAMP > DATEADD('MINUTE', -5, CURRENT_TIMESTAMP())
        HAVING COUNT(*) > 10
    ))
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'HOL_EMAIL_INTEGRATION',
            'data-engineering@jpmc.com',
            'ALERT: High DML Error Rate Detected',
            'More than 10 DML errors detected in TRANSACTIONS_VALIDATED in the last 5 minutes. Please investigate.'
        );

-- NOTE: Email integration requires setup. For demo, we'll use a Task-based approach instead:
CREATE OR REPLACE TASK TASK_ERROR_MONITOR
    WAREHOUSE = HOL_XS_WH
    SCHEDULE = '5 MINUTE'
    COMMENT = 'Monitor error tables and log alerts'
AS
    INSERT INTO HOL_LAB.ERROR_ALERT_LOG (ALERT_TIME, TABLE_NAME, ERROR_COUNT, DETAILS)
    SELECT 
        CURRENT_TIMESTAMP(),
        'TRANSACTIONS_VALIDATED',
        COUNT(*),
        ARRAY_AGG(OBJECT_CONSTRUCT('error_code', ERROR_CODE, 'message', ERROR_MESSAGE))
    FROM SNOWFLAKE.CORE.GET_ERRORS('JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TRANSACTIONS_VALIDATED')
    WHERE ERROR_TIMESTAMP > DATEADD('MINUTE', -5, CURRENT_TIMESTAMP())
    HAVING COUNT(*) > 5;

In [ ]:
-- Create the alert log table
CREATE OR REPLACE TABLE ERROR_ALERT_LOG (
    ALERT_TIME TIMESTAMP,
    TABLE_NAME VARCHAR,
    ERROR_COUNT INT,
    DETAILS VARIANT
);

-- For the demo, check COPY INTO history for observability
SELECT *
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'TRANSACTIONS',
    START_TIME => DATEADD(HOUR, -24, CURRENT_TIMESTAMP())
))
ORDER BY LAST_LOAD_TIME DESC
LIMIT 10;

## Step 6: Pipeline Observability Dashboard Query

A comprehensive query for monitoring your entire ingestion pipeline health.

In [ ]:
-- =============================================================================
-- PIPELINE HEALTH DASHBOARD
-- =============================================================================

-- Check Dynamic Table refresh health
SELECT 
    NAME,
    REFRESH_ACTION,
    STATE,
    STATE_MESSAGE,
    REFRESH_START_TIME,
    REFRESH_END_TIME,
    DATEDIFF('second', REFRESH_START_TIME, REFRESH_END_TIME) AS DURATION_SECONDS
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE REFRESH_START_TIME > DATEADD(HOUR, -24, CURRENT_TIMESTAMP())
ORDER BY REFRESH_START_TIME DESC
LIMIT 20;

In [ ]:
-- Task execution health
SELECT 
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SEC,
    RETURN_VALUE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(HOUR, -24, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 20;

## Summary

| Feature | Purpose | Key Benefit |
|---------|---------|-------------|
| **Error Tables** | Capture row-level DML failures | Continue loading good rows, log bad ones |
| **Alerts** | Threshold-based notifications | Proactive monitoring without manual checks |
| **COPY_HISTORY** | Track file ingestion results | Audit trail for all data loads |
| **DT Refresh History** | Monitor transformation pipeline | Identify slow or failing refreshes |
| **Task History** | Track scheduled job execution | Ensure orchestration runs successfully |

---

**Next:** Proceed to `07_Storage_Lifecycle_Iceberg_CoCo.ipynb` for data archival, Iceberg interoperability, and Cortex Code.